# 03 — Feature Extraction

Explore the epoch-level feature store produced by `scripts/ingest_features.py`.

**CLI alternative** (saves PNGs + JSON to `results/qc/features/`):

```bash
python scripts/inspect_qc.py features
python scripts/inspect_qc.py spot-check --dataset 2 --subject 1
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import pandas as pd

from config import FEATURE_COLUMNS, PARQUET_COMBINED_FILE
from util.io import load_features_df
from util.qc import feature_summary, plot_feature_report, spot_check_features

%matplotlib inline

In [ ]:
if not Path(PARQUET_COMBINED_FILE).exists():
    raise FileNotFoundError(
        f"{PARQUET_COMBINED_FILE} not found. Run: python scripts/ingest_features.py --all-datasets --all"
    )

df = load_features_df()
print(f"Rows: {len(df)}, subjects: {df['participant_id'].nunique()}")
df[FEATURE_COLUMNS + ['label']].describe()

In [ ]:
summary = feature_summary(df)
print(json.dumps({k: v for k, v in summary.items() if k != "features_by_label"}, indent=2))

missing = pd.Series(summary["missing_values"])
if missing.any():
    print("\nMissing values:")
    print(missing[missing > 0])
else:
    print("\nNo missing values in feature columns.")

In [ ]:
from IPython.display import Image, display

report_path = plot_feature_report(df, show=False)
print(f"Saved feature QC plots to {report_path.parent}")

for name in ["feature_distributions.png", "epochs_per_subject.png", "feature_correlation.png"]:
    p = report_path.parent / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

## Spot-check: re-extract one subject

Re-runs preprocessing + feature extraction for a single subject and compares against stored parquet values. Large differences may indicate a pipeline change or ingest error.

In [ ]:
SPOT_DATASET = 2
SPOT_SUBJECT = 1

comparison = spot_check_features(SPOT_DATASET, SPOT_SUBJECT, df=df)
comparison.head(10)